# EasyOCR Evaluation on Synthetic Handwritten Data

This notebook evaluates EasyOCR performance on synthetic handwritten German nursing notes.

**Metrics Calculated:**
- Accuracy (sequence similarity)
- CER (Character Error Rate)
- WER (Word Error Rate)
- NED (Normalized Edit Distance)
- Vocabulary Precision (unique word overlap)
- Vocabulary Recall (unique word overlap)
- Vocabulary F1-Score (unique word overlap)

**Output:**
- Individual results: `results/metrics/easyocr/individual/`
- Summary CSV: `results/metrics/easyocr/summary.csv`
- Summary JSON: `results/metrics/easyocr/summary.json`
- Detailed results JSON: `results/metrics/easyocr/detailed_results.json`

## 1. Setup and Imports

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import easyocr
import json
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# Import our custom metrics module
from src.evaluation.metrics import calculate_all_metrics, format_metrics_display

print("✓ All imports successful")
print(f"Project root: {project_root}")

/home/bashar/bachelor-thesis-nursing-agents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
Project root: /home/bashar/bachelor-thesis-nursing-agents


## 2. Configuration

In [2]:
# Paths
IMAGES_DIR = project_root / "data" / "synthetic" / "output" / "images"
LABELS_DIR = project_root / "data" / "synthetic" / "output" / "labels"
RESULTS_DIR = project_root / "results" / "metrics" / "easyocr"
INDIVIDUAL_RESULTS_DIR = RESULTS_DIR / "individual"

# Create output directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INDIVIDUAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# OCR Configuration
LANGUAGES = ['de']  # German
USE_GPU = True

# ==================================================================================
# EVALUATION STRATEGY - Choose one:
# ==================================================================================
# Option A: Use fixed test set (RECOMMENDED for fair comparison)
USE_TEST_SET = True
TEST_SET_FILE = project_root / "data" / "test_set_600.json"  # or test_set_500.json, test_set_complete.json

# Option B: Random sampling (NOT recommended for comparing OCR methods)
# USE_TEST_SET = False
# LIMIT_SAMPLES = None  # Set to a number (e.g., 10) for testing, None for all images
# ==================================================================================

print(f"Images directory: {IMAGES_DIR}")
print(f"Labels directory: {LABELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Using GPU: {USE_GPU}")
print(f"Languages: {LANGUAGES}")

if USE_TEST_SET:
    if TEST_SET_FILE.exists():
        print(f"✓ Using FIXED TEST SET: {TEST_SET_FILE}")
        print("  (Ensures fair comparison across OCR methods)")
    else:
        print(f"⚠ TEST SET FILE NOT FOUND: {TEST_SET_FILE}")
        print("  Run: python scripts/create_test_set.py --strategy balanced-600 --output data/test_set_600.json")
        USE_TEST_SET = False
        LIMIT_SAMPLES = None
else:
    print(f"⚠ Using random sampling: {LIMIT_SAMPLES if LIMIT_SAMPLES else 'All images'}")
    print("  Note: This may not be fair for comparing different OCR methods!")

Images directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/images
Labels directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/labels
Results directory: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/easyocr
Using GPU: True
Languages: ['de']
✓ Using FIXED TEST SET: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
  (Ensures fair comparison across OCR methods)


### 📌 Important: Fair Evaluation Strategy

**Problem**: Randomly selecting samples can lead to unfair comparisons between OCR methods.

**Solution**: Use a **fixed stratified test set** that ensures:
- Balanced representation across all fonts
- Coverage of all scenarios
- Same samples used for ALL OCR methods

**First, create the test set (run once):**
```bash
# Option 1: Complete coverage (1200 samples - 1 per font-scenario)
python scripts/create_test_set.py --strategy complete --output data/test_set_complete.json

# Option 2: Balanced 600 samples
python scripts/create_test_set.py --strategy balanced-600 --output data/test_set_600.json

# Option 3: Stratified 500 samples
python scripts/create_test_set.py --strategy stratified-500 --output data/test_set_500.json --seed 42
```

Then configure below to use the test set:

## 3. Initialize EasyOCR Reader

In [3]:
print("Initializing EasyOCR reader...")
print("This may take a moment on first run (downloading models)...")

reader = easyocr.Reader(LANGUAGES, gpu=USE_GPU)

print("✓ EasyOCR reader initialized successfully")

Initializing EasyOCR reader...
This may take a moment on first run (downloading models)...
✓ EasyOCR reader initialized successfully


## 4. Load Image and Label Pairs

In [4]:
# Load image-label pairs based on evaluation strategy
valid_pairs = []

if USE_TEST_SET and TEST_SET_FILE.exists():
    # Load from fixed test set
    print(f"Loading test set from: {TEST_SET_FILE}")
    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        test_set_data = json.load(f)
    
    print(f"Test set contains {test_set_data['test_set_size']} samples")
    
    for sample in test_set_data['samples']:
        img_path = project_root / sample['image_path']
        label_path = project_root / sample['label_path']
        
        if img_path.exists() and label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Missing files for {sample['sample_id']}")
    
    print(f"✓ Loaded {len(valid_pairs)} samples from FIXED TEST SET")
    print(f"  (Same samples will be used for all OCR methods)")

else:
    # Fall back to all images or limited sampling
    print("Loading all images from directory...")
    image_files = sorted(list(IMAGES_DIR.glob("*.png")))
    
    if 'LIMIT_SAMPLES' in globals() and LIMIT_SAMPLES:
        image_files = image_files[:LIMIT_SAMPLES]
    
    print(f"Found {len(image_files)} images")
    
    # Verify corresponding label files exist
    for img_path in image_files:
        label_path = LABELS_DIR / f"{img_path.stem}.json"
        if label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Warning: No label file found for {img_path.name}")
    
    print(f"✓ {len(valid_pairs)} valid image-label pairs ready")
    if not USE_TEST_SET:
        print(f"⚠ WARNING: Not using fixed test set - may not be fair for OCR comparison!")

Loading test set from: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
Test set contains 559 samples
✓ Loaded 559 samples from FIXED TEST SET
  (Same samples will be used for all OCR methods)


## 5. Batch Processing with OCR and Evaluation

In [5]:
results_list = []
failed_samples = []

print(f"\n{'=' * 80}")
print("STARTING BATCH OCR PROCESSING")
print(f"{'=' * 80}\n")

for img_path, label_path in tqdm(valid_pairs, desc="Processing images"):
    sample_id = img_path.stem
    
    try:
        # Load ground truth
        with open(label_path, 'r', encoding='utf-8') as f:
            label_data = json.load(f)
        
        expected_text = label_data.get('full_text', '')
        
        # Run OCR
        ocr_results = reader.readtext(str(img_path))
        
        # Extract text from OCR results
        predicted_text = " ".join([text for (bbox, text, conf) in ocr_results])
        
        # Calculate all metrics
        metrics = calculate_all_metrics(expected_text, predicted_text)
        
        # Prepare result dictionary
        result = {
            'sample_id': sample_id,
            'image_path': str(img_path.relative_to(project_root)),
            'expected_text': expected_text,
            'predicted_text': predicted_text,
            'expected_length': len(expected_text),
            'predicted_length': len(predicted_text),
            'num_detections': len(ocr_results),
            **metrics,  # Add all metrics
            'detections': [
                {
                    'text': text,
                    'confidence': float(conf),
                    'bbox': [[int(x), int(y)] for x, y in bbox]
                }
                for bbox, text, conf in ocr_results
            ]
        }
        
        results_list.append(result)
        
        # Save individual result
        individual_result_path = INDIVIDUAL_RESULTS_DIR / f"{sample_id}_result.json"
        with open(individual_result_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
    except Exception as e:
        print(f"✗ Error processing {sample_id}: {str(e)}")
        failed_samples.append({'sample_id': sample_id, 'error': str(e)})

print(f"\n{'=' * 80}")
print(f"✓ Successfully processed: {len(results_list)} samples")
print(f"✗ Failed: {len(failed_samples)} samples")
print(f"{'=' * 80}")


STARTING BATCH OCR PROCESSING



Processing images: 100%|██████████| 559/559 [05:49<00:00,  1.60it/s]


✓ Successfully processed: 559 samples
✗ Failed: 0 samples


## 6. Create Summary Statistics

In [6]:
if len(results_list) > 0:
    # Create DataFrame for analysis
    df_results = pd.DataFrame([
        {
            'sample_id': r['sample_id'],
            'accuracy': r['accuracy'],
            'cer': r['cer'],
            'wer': r['wer'],
            'ned': r['ned'],
            'vocab_precision': r['vocab_precision'],
            'vocab_recall': r['vocab_recall'],
            'vocab_f1': r['vocab_f1'],
            'expected_length': r['expected_length'],
            'predicted_length': r['predicted_length'],
            'num_detections': r['num_detections']
        }
        for r in results_list
    ])
    
    # Calculate summary statistics
    summary_stats = {
        'method': 'EasyOCR',
        'languages': LANGUAGES,
        'num_samples': len(results_list),
        'timestamp': datetime.now().isoformat(),
        'metrics': {
            'accuracy': {
                'mean': float(df_results['accuracy'].mean()),
                'std': float(df_results['accuracy'].std()),
                'min': float(df_results['accuracy'].min()),
                'max': float(df_results['accuracy'].max()),
                'median': float(df_results['accuracy'].median())
            },
            'cer': {
                'mean': float(df_results['cer'].mean()),
                'std': float(df_results['cer'].std()),
                'min': float(df_results['cer'].min()),
                'max': float(df_results['cer'].max()),
                'median': float(df_results['cer'].median())
            },
            'wer': {
                'mean': float(df_results['wer'].mean()),
                'std': float(df_results['wer'].std()),
                'min': float(df_results['wer'].min()),
                'max': float(df_results['wer'].max()),
                'median': float(df_results['wer'].median())
            },
            'ned': {
                'mean': float(df_results['ned'].mean()),
                'std': float(df_results['ned'].std()),
                'min': float(df_results['ned'].min()),
                'max': float(df_results['ned'].max()),
                'median': float(df_results['ned'].median())
            },
            'vocab_precision': {
                'mean': float(df_results['vocab_precision'].mean()),
                'std': float(df_results['vocab_precision'].std()),
                'min': float(df_results['vocab_precision'].min()),
                'max': float(df_results['vocab_precision'].max()),
                'median': float(df_results['vocab_precision'].median())
            },
            'vocab_recall': {
                'mean': float(df_results['vocab_recall'].mean()),
                'std': float(df_results['vocab_recall'].std()),
                'min': float(df_results['vocab_recall'].min()),
                'max': float(df_results['vocab_recall'].max()),
                'median': float(df_results['vocab_recall'].median())
            },
            'vocab_f1': {
                'mean': float(df_results['vocab_f1'].mean()),
                'std': float(df_results['vocab_f1'].std()),
                'min': float(df_results['vocab_f1'].min()),
                'max': float(df_results['vocab_f1'].max()),
                'median': float(df_results['vocab_f1'].median())
            }
        }
    }
    
    print("Summary Statistics:")
    print("=" * 80)
    for metric_name, stats in summary_stats['metrics'].items():
        print(f"\n{metric_name.upper()}:")
        print(f"  Mean:   {stats['mean']:.2f}%")
        print(f"  Median: {stats['median']:.2f}%")
        print(f"  Std:    {stats['std']:.2f}%")
        print(f"  Range:  [{stats['min']:.2f}%, {stats['max']:.2f}%]")
    print("=" * 80)
else:
    print("⚠ No results to summarize")

Summary Statistics:

ACCURACY:
  Mean:   13.62%
  Median: 9.07%
  Std:    13.00%
  Range:  [0.00%, 75.67%]

CER:
  Mean:   60.67%
  Median: 65.49%
  Std:    19.60%
  Range:  [12.53%, 100.00%]

WER:
  Mean:   94.45%
  Median: 98.15%
  Std:    11.59%
  Range:  [39.22%, 125.45%]

NED:
  Mean:   60.66%
  Median: 65.49%
  Std:    19.60%
  Range:  [12.53%, 100.00%]

VOCAB_PRECISION:
  Mean:   12.88%
  Median: 6.67%
  Std:    14.73%
  Range:  [0.00%, 65.31%]

VOCAB_RECALL:
  Mean:   13.06%
  Median: 6.25%
  Std:    15.49%
  Range:  [0.00%, 68.09%]

VOCAB_F1:
  Mean:   12.92%
  Median: 6.11%
  Std:    15.10%
  Range:  [0.00%, 66.67%]


## 7. Save Results

In [7]:
if len(results_list) > 0:
    # Save CSV (for easy analysis in Excel/Python)
    csv_path = RESULTS_DIR / "summary.csv"
    df_results.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✓ Saved CSV summary: {csv_path}")
    
    # Save summary JSON
    summary_json_path = RESULTS_DIR / "summary.json"
    with open(summary_json_path, 'w', encoding='utf-8') as f:
        json.dump(summary_stats, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved JSON summary: {summary_json_path}")
    
    # Save detailed results (all data including OCR detections)
    detailed_json_path = RESULTS_DIR / "detailed_results.json"
    with open(detailed_json_path, 'w', encoding='utf-8') as f:
        json.dump(results_list, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved detailed results: {detailed_json_path}")
    
    # Save failed samples if any
    if failed_samples:
        failed_path = RESULTS_DIR / "failed_samples.json"
        with open(failed_path, 'w', encoding='utf-8') as f:
            json.dump(failed_samples, f, ensure_ascii=False, indent=2)
        print(f"⚠ Saved failed samples log: {failed_path}")
    
    print(f"\n{'=' * 80}")
    print("ALL RESULTS SAVED SUCCESSFULLY")
    print(f"{'=' * 80}")
    print(f"\nResults location: {RESULTS_DIR}")
    print(f"  - summary.csv ({len(df_results)} rows)")
    print(f"  - summary.json (aggregate statistics)")
    print(f"  - detailed_results.json (full OCR output)")
    print(f"  - individual/ ({len(results_list)} files)")
else:
    print("⚠ No results to save")

✓ Saved CSV summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/easyocr/summary.csv
✓ Saved JSON summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/easyocr/summary.json
✓ Saved detailed results: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/easyocr/detailed_results.json

ALL RESULTS SAVED SUCCESSFULLY

Results location: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/easyocr
  - summary.csv (559 rows)
  - summary.json (aggregate statistics)
  - detailed_results.json (full OCR output)
  - individual/ (559 files)


## 8. Quick Analysis and Visualization

In [8]:
if len(results_list) > 0:
    # Display top 5 best and worst performing samples
    print("\n" + "=" * 80)
    print("TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    top_5 = df_results.nlargest(5, 'vocab_f1')
    print(top_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))
    
    print("\n" + "=" * 80)
    print("TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    bottom_5 = df_results.nsmallest(5, 'vocab_f1')
    print(bottom_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))
    
    # Basic statistics
    print("\n" + "=" * 80)
    print("BASIC STATISTICS")
    print("=" * 80)
    print(f"Samples with accuracy > 90%: {(df_results['accuracy'] > 90).sum()} ({(df_results['accuracy'] > 90).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 80%: {(df_results['accuracy'] > 80).sum()} ({(df_results['accuracy'] > 80).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 70%: {(df_results['accuracy'] > 70).sum()} ({(df_results['accuracy'] > 70).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with CER < 10%: {(df_results['cer'] < 10).sum()} ({(df_results['cer'] < 10).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with Vocabulary F1 > 80%: {(df_results['vocab_f1'] > 80).sum()} ({(df_results['vocab_f1'] > 80).sum() / len(df_results) * 100:.1f}%)")
else:
    print("⚠ No data for analysis")


TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy       cer       wer  vocab_f1
sample_0211 75.668073 15.555556 39.215686 66.666667
sample_1678 75.000000 15.319149 48.484848 64.615385
sample_0096 42.937853 17.293233 52.941176 63.768116
sample_0379 71.637427 14.161850 45.454545 60.606061
sample_2000 30.217670 12.846348 46.774194 58.407080

TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy       cer        wer  vocab_f1
sample_0319 55.050505 56.603774 100.000000       0.0
sample_0020 11.954023 50.638298 100.000000       0.0
sample_0302 12.500000 85.502959 100.000000       0.0
sample_0089 24.875622 78.571429 100.000000       0.0
sample_1354  2.072539 71.301775 104.444444       0.0

BASIC STATISTICS
Samples with accuracy > 90%: 0 (0.0%)
Samples with accuracy > 80%: 0 (0.0%)
Samples with accuracy > 70%: 3 (0.5%)
Samples with CER < 10%: 0 (0.0%)
Samples with Vocabulary F1 > 80%: 0 (0.0%)


## 9. Example: Detailed View of a Sample

Run this cell to see detailed results for a specific sample:

In [9]:
# Change this to any sample_id you want to inspect
SAMPLE_TO_INSPECT = results_list[0]['sample_id'] if len(results_list) > 0 else None

if SAMPLE_TO_INSPECT:
    sample_result = next((r for r in results_list if r['sample_id'] == SAMPLE_TO_INSPECT), None)
    
    if sample_result:
        print("=" * 80)
        print(f"DETAILED RESULTS FOR: {SAMPLE_TO_INSPECT}")
        print("=" * 80)
        
        print(f"\n📄 Expected Text ({sample_result['expected_length']} chars):")
        print("-" * 80)
        print(sample_result['expected_text'])
        
        print(f"\n🤖 Predicted Text ({sample_result['predicted_length']} chars):")
        print("-" * 80)
        print(sample_result['predicted_text'])
        
        print(f"\n{format_metrics_display(sample_result)}")
        
        print(f"\n{'=' * 80}")
        print(f"DETECTED TEXT REGIONS ({sample_result['num_detections']} detections)")
        print("=" * 80)
        for i, detection in enumerate(sample_result['detections'], 1):
            print(f"\n{i}. '{detection['text']}'")
            print(f"   Confidence: {detection['confidence']:.2f}")
else:
    print("⚠ No samples available for inspection")

DETAILED RESULTS FOR: sample_0246

📄 Expected Text (1069 chars):
--------------------------------------------------------------------------------
Patient: Albert Schuster Datum: 29.10.2025 29.10.2025, Hr. Schuster, 92 J., Zi. 32, PG 5
KOMPLETTE IMMOBILITÄT, bettlägerig seit Schlaganfall 2024. Grundpflege 7:00 Uhr: -
Ganzwaschwäsche im Bett, Pat. kann nicht mithelfen - Inkontinenzversorgung (Windel, Bettschutz)
- Katheter suprapubisch, Urin klar-gelb, ca. 800ml/24h - Mundpflege intensiv Lagerungsplan:
7:00 rechts - 10:00 links - 13:00 rechts - 16:00 links - 19:00 rechts Anti-Dekubitus-Matratze
im Einsatz. DEKUBITUS-STATUS: - Sakral Grad 2, ca. 3x4 cm, Wundversorgung mit HydroClean
- Beide Fersen freigelagert mit Gelkissen - Haut ansonsten intakt Ernährung: - Vollkost püriert
über Löffel - Frühstück: 5 Löffel Brei - Mittag: ca. 10 Löffel Gemüsebrei, 3 Löffel Kartoffelpüree -
Trinkmenge: 600ml (in kleinen Schlucken) Kommunikation: Pat. reagiert mit Augenkontakt, kann
nicht sprechen. Verst

---
## ℹ️ Usage Instructions

### Quick Start (Step by Step)
1. **Run Cell 3**: Initialize imports and verify paths
2. **Run Cell 5**: Configure paths and settings (optional: set `LIMIT_SAMPLES = 10` for quick testing)
3. **Run Cell 7**: Initialize EasyOCR reader (takes ~30 seconds first time)
4. **Run Cell 9**: Load and validate image-label pairs
5. **Run Cell 11**: **Main processing** - Process all images with OCR (this may take a while!)
6. **Run Cell 13**: Calculate summary statistics
7. **Run Cell 15**: Save all results
8. **Run Cells 17 & 19**: View quick analysis and sample details

### Expected Output Files
- `results/metrics/easyocr/summary.csv` - All samples with metrics
- `results/metrics/easyocr/summary.json` - Aggregate statistics
- `results/metrics/easyocr/detailed_results.json` - Full OCR output
- `results/metrics/easyocr/individual/` - Per-sample JSON files

### For Later Analysis
Use the analysis script: `python results/metrics/easyocr/analyze_results.py`

Or in Python:
```python
from results.metrics.easyocr.analyze_results import load_results, print_summary
df = load_results('results/metrics/easyocr')
print_summary(df)
```

### Testing First
Set `LIMIT_SAMPLES = 10` in Cell 5 to test on a small subset before processing all 500 images.

---

## Next Steps

For LLM processing of these OCR results, use the standalone LLM pipeline:

```bash
# Process EasyOCR results with Qwen
python scripts/run_llm_pipeline.py --ocr-method easyocr --model qwen-7b

# Evaluate LLM outputs
python scripts/evaluate_llm.py --input results/metrics/easyocr/llm_outputs/.../all_llm_results.json
```

See `docs/STANDALONE_LLM_PIPELINE.md` for details.

---

## Next Steps: LLM Processing

For LLM processing of these OCR results, use the standalone LLM pipeline:

```bash
# Process EasyOCR results with Qwen 7B
python scripts/run_llm_pipeline.py --ocr-method easyocr --model qwen-7b

# Or test with smaller model first (10 samples)
python scripts/run_llm_pipeline.py --ocr-method easyocr --model qwen-1.5b --limit 10

# Use different models (any HuggingFace model!)
python scripts/run_llm_pipeline.py --ocr-method easyocr --model meta-llama/Llama-3.1-8B-Instruct
python scripts/run_llm_pipeline.py --ocr-method easyocr --model mistralai/Mistral-7B-Instruct-v0.3

# Evaluate LLM outputs
python scripts/evaluate_llm.py --input results/metrics/easyocr/llm_outputs/qwen2.5-7b-instruct_nursing-summary/all_llm_results.json
```

**Available Models:**
- `qwen-1.5b`, `qwen-3b`, `qwen-7b`, `qwen-14b` (presets)
- Any HuggingFace model: `meta-llama/Llama-3.1-8B-Instruct`, `mistralai/Mistral-7B-Instruct-v0.3`, etc.

**Available Prompts:**
- `nursing_summary` (default) - Summarize + extract features + suggestions
- `critical_info_extraction` - Extract only critical medical info
- `care_plan_generation` - Generate care plans
- `risk_assessment` - Assess patient risks
- `simple_summary` - Short 2-3 sentence summary

See the [complete guide](../docs/STANDALONE_LLM_PIPELINE.md) for more details.